# CCT — Plate OCR (FastPlateOCR)

Notebook train CCT (compact convolutional transformer) cho OCR biển số.
Dùng FastPlateOCR ≥ 1.1.0 (đã pin). Input là dataset CSV với
`image_path,plate_text` đã qua R8 duyệt.

CCT CLI `--weights-path` KHÔNG phải resume đầy đủ — chỉ nạp weights và tạo
optimizer mới. Resume đầy đủ cần load `last.keras` qua `load_keras_model` với
config/manifest/batch và `initial_epoch`. Cell bên dưới có 2 lựa chọn.

Output cuối: ONNX (uint8/channels_last) + YAML config + manifest + hash.
Round-trip: test OCR trên ảnh mẫu cùng crop với EasyOCR baseline.


In [ ]:
import os
# Kaggle environment: pin Ultralytics, torch, fast_plate_ocr theo kaggle image.
# KHÔNG ép nâng cấp Torch local — pin đúng version đang dùng (YOLO11 cần ≥8.4).
os.environ.setdefault("YOLO_CONFIG_DIR", "/kaggle/working/yolo_cfg")
os.environ.setdefault("ULTRALYTICS_RUNS_DIR", "/kaggle/working/runs")
os.environ.setdefault("ULTRALYTICS_OFFLINE", "True")
os.environ.setdefault("ULTRALYTICS_TELEMETRY", "False")
# Tắt mọi plotting/caching Ultralytics
os.environ.setdefault("YOLO_VERBOSE", "False")
import torch
print(f"torch={torch.__version__} cuda={torch.cuda.is_available()}")


In [ ]:
# Pin package version cho CCT — fast_plate_ocr 1.1.0 đã verify ở DATA_REPORT.
from importlib import metadata
for pkg in ("fast-plate-ocr", "onnxruntime"):
    try:
        print(pkg, "=", metadata.version(pkg))
    except metadata.PackageNotFoundError:
        print(pkg, "= MISSING")
print("python:", __import__("sys").version.split()[0])


In [ ]:
# Kaggle /kaggle/working là output dir. Tất cả artifact phải vào đây.
import os
from pathlib import Path
WORKDIR = Path(os.environ.get("WORKDIR", "/kaggle/working"))
WORKDIR.mkdir(parents=True, exist_ok=True)
print("workdir:", WORKDIR)


In [ ]:
def preflight(audit_dir: str) -> dict:
    """Kiểm tra manifest + hash + group-disjoint trước khi train.

    Trả về dict với 'ok' bool + 'pending_data' nếu thiếu.
    Raise nếu hash khác (file đã bị sửa sau export).
    """
    import hashlib
    import os
    import json

    manifest_path = os.path.join(audit_dir, "manifest.json")
    if not os.path.isfile(manifest_path):
        return {"ok": False, "pending_data": True,
                "reason": f"manifest.json không có tại {audit_dir}"}
    with open(manifest_path, "r", encoding="utf-8") as f:
        manifest = json.load(f)
    # Manifest.sha256 đi kèm file đã verify ở R8; ở đây chỉ in lại
    digest_path = os.path.join(audit_dir, "manifest.sha256")
    if os.path.isfile(digest_path):
        recorded = open(digest_path, encoding="ascii").read().strip()
        actual = hashlib.sha256(open(manifest_path, "rb").read()).hexdigest()
        if recorded != actual:
            raise ValueError(
                f"manifest hash mismatch: file={recorded} vs recompute={actual}"
            )
    counts = manifest.get("counts", {})
    needed = {"train", "val", "test"}
    if not needed.issubset(counts):
        return {"ok": False, "pending_data": True,
                "reason": f"split thiếu: cần {needed}, có {set(counts)}"}
    if any(v == 0 for v in counts.values()):
        return {"ok": False, "pending_data": True,
                "reason": f"split có count=0: {counts}"}
    if manifest.get("holdout_status") != "human_curated_group_disjoint":
        return {"ok": False, "pending_data": True,
                "reason": f"holdout_status chưa phải human_curated: {manifest.get('holdout_status')}"}
    return {"ok": True, "manifest": manifest}


print("preflight helpers loaded")


In [ ]:
import os
from pathlib import Path
OCR_SLUG = os.environ.get("OCR_DATASET_SLUG", "plate-ocr-v1")
ocr_dir = os.path.join(KAGGLE_INPUT, OCR_SLUG)
pf = preflight(ocr_dir)
if not pf["ok"]:
    raise SystemExit(f"ocr pending_data: {pf.get('reason')}")
print("ocr manifest ok:", pf["manifest"]["counts"])
# CCT CSV theo split
for split in ("train", "val", "test"):
    csv_path = Path(ocr_dir) / f"{split}.csv"
    if not csv_path.is_file():
        raise FileNotFoundError(f"thiếu {split}.csv")
print("ocr csv:", [p.name for p in Path(ocr_dir).glob("*.csv")])


In [ ]:
# Resume đầy đủ: load compiled last.keras + initial_epoch
from pathlib import Path
# Chuẩn bị dataset theo format FastPlateOCR: image_path,plate_text
import shutil
cct_root = WORKDIR / "cct_dataset"
(cct_root / "train").mkdir(parents=True, exist_ok=True)
(cct_root / "val").mkdir(parents=True, exist_ok=True)
(cct_root / "test").mkdir(parents=True, exist_ok=True)
import pandas as pd
for split in ("train", "val", "test"):
    df = pd.read_csv(Path(ocr_dir) / f"{split}.csv")
    for _, row in df.iterrows():
        src_img = Path(ocr_dir) / row["image_path"]
        dst_img = cct_root / split / Path(row["image_path"]).name
        if src_img.is_file() and not dst_img.exists():
            shutil.copy(src_img, dst_img)
    (cct_root / f"{split}.csv").write_text(
        df.to_csv(index=False), encoding="utf-8",
    )

# Train CCT CLI (đầy đủ): compile + last.keras + initial_epoch
import subprocess
resume_ckpt = WORKDIR / "cct_last.keras"
cmd = [
    "cct-train",
    "--train-csv", str(cct_root / "train.csv"),
    "--val-csv", str(cct_root / "val.csv"),
    "--img-height", "64",
    "--img-width", "128",
    "--max-plate-slots", "10",
    "--alphabet", "0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ_",
    "--pad-char", "_",
    "--batch-size", "32",
    "--epochs", str(int(os.environ.get("EPOCHS", "60"))),
    "--save-dir", str(WORKDIR / "cct_run"),
]
if resume_ckpt.is_file():
    # Resume đầy đủ: dùng Keras load_model với compiled state
    cmd += ["--resume-keras", str(resume_ckpt)]
print("running:", " ".join(cmd))
subprocess.run(cmd, check=True)
print("cct train done")


In [ ]:
# Lưu ý: --weights-path CHỈ nạp weights + tạo optimizer mới. KHÔNG phải resume đầy đủ.
# Dùng khi train fresh (không có compiled checkpoint). KHÔNG dùng khi muốn tiếp tục
# training từ epoch trước mà giữ optimizer/regularizer.
import os
weights_path = os.environ.get("CCT_WEIGHTS_PATH", "")
if weights_path:
    print(f"--weights-path={weights_path} (NOTE: this is NOT full resume)")


In [ ]:
# Export CCT ONNX + YAML + manifest
import hashlib
import json
import shutil
from importlib import metadata
from pathlib import Path

# Sau khi train, CCT CLI sinh ra .keras. Tìm và convert ONNX.
keras_ckpt = WORKDIR / "cct_run" / "last.keras"
if not keras_ckpt.is_file():
    raise FileNotFoundError(f"cct last.keras không có tại {keract}")

# CCT CLI export ONNX với uint8 + simplify (đọc docs FastPlateOCR)
import subprocess
onnx_out = WORKDIR / "plate_ocr_v1.onnx"
cmd = [
    "cct-export",
    "--weights", str(keract),
    "--output", str(onnx_out),
    "--opset", "12",
    "--channels-last",
]
print("running:", " ".join(cmd))
subprocess.run(cmd, check=True)

# Copy plate_config.yaml từ dataset portable (đã verify ở R8)
src_yaml = Path(ocr_dir) / "plate_config.yaml"
dst_yaml = WORKDIR / "plate_ocr_v1.yaml"
shutil.copy(src_yaml, dst_yaml)

artifact = {
    "schema_version": 1,
    "task": "plate_ocr",
    "engine": "fast_plate_ocr.cct",
    "package_versions": {
        "fast-plate-ocr": metadata.version("fast-plate-ocr"),
        "onnxruntime": metadata.version("onnxruntime"),
        "torch": torch.__version__,
    },
    "onnx": str(onnx_out),
    "yaml": str(dst_yaml),
    "config_sha256": hashlib.sha256(dst_yaml.read_bytes()).hexdigest(),
    "holdout_status": pf["manifest"]["holdout_status"],
    "source_manifest_sha256": pf["manifest"].get("source_manifest_sha256", ""),
}
out_path = WORKDIR / "artifact_plate_ocr.json"
out_path.write_text(json.dumps(artifact, indent=2, ensure_ascii=False), encoding="utf-8")
sha = hashlib.sha256(out_path.read_bytes()).hexdigest()
(WORKDIR / "artifact_plate_ocr.sha256").write_text(sha, encoding="ascii")
print("artifact:", out_path, sha)


In [ ]:
# Round-trip smoke: OCR trên ảnh test, so sánh với ground truth
from pathlib import Path
import csv
from fast_plate_ocr import LicensePlateRecognizer
reader = LicensePlateRecognizer(
    onnx_model_path=str(WORKDIR / "plate_ocr_v1.onnx"),
    plate_config_path=str(WORKDIR / "plate_ocr_v1.yaml"),
    device="cpu",
)
gt_path = Path(ocr_dir) / "test.csv"
hits, total = 0, 0
with gt_path.open(encoding="utf-8") as f:
    reader_csv = csv.DictReader(f)
    for row in list(reader_csv)[:10]:  # smoke 10 ảnh đầu
        img = Path(ocr_dir) / row["image_path"]
        if not img.is_file():
            continue
        pred = reader.run(str(img))[0].plate
        gt = row["plate_text"].strip()
        total += 1
        if pred == gt:
            hits += 1
print(f"cct smoke: {hits}/{total} exact match")
assert total > 0, "no test images found"
print("cct smoke ok")
